# From URDF to a Semantic Digital Twin

*IJCAI 2026 workshop — hands-on tutorial (150 minutes)*

A URDF tells a robot **where** every link is. It does not tell the robot **what** any of
them is. `<link name="iai_fridge_door"/>` is a name a human chose; to the robot it is a
rigid body on a revolute joint, indistinguishable from a cabinet door, a car door, or a
telescopic mast.

So "take the milk out of the fridge" is not a question a URDF can answer. Before a robot can
even think about reaching for the milk, it needs to know: which body is *the fridge*? Does it
have a door? Which way does the door open, and how far? Is there a handle, and where is it?
None of that is in the geometry — it has to be *added*.

This tutorial closes that gap using the
[Semantic Digital Twin](https://github.com/cram2/cognitive_robot_abstract_machine), a world
model that carries geometry, kinematics **and** meaning in one structure. By the end you will
be able to load a kitchen, find its fridge, and open the door standing between the robot and
the milk — everything a robot needs to *know* before it can *act*.

> **Scope.** This notebook is about the world model itself: bodies, connections, and the
> semantic annotations layered on top of them. It deliberately stops before motion planning,
> reaching, or grasping — those build on exactly this representation, one layer up, and are
> the subject of a later notebook in this series.

| § | What we do |
|---|---|
| 1 | Load a kitchen URDF and watch two reasonable heuristics both get it wrong |
| 2 | Say what things *are*: build a dresser from typed semantic annotations |
| 3 | Let the `WorldReasoner` find the drawers itself — and see what it still misses |
| 4 | Find the fridge, and open the door standing between the robot and the milk |
| 5 | Bring a robot into the world and see it standing in the kitchen |
| 6 | Put it all together on a second kitchen, on your own |

**Prerequisites:** Python, and having seen a URDF before. No prior knowledge of CRAM,
ontologies, or rule-based reasoning is assumed.

## 0. Setup

> **Check your kernel.** The top right of this notebook must say **CRAM**. If it says
> anything else, use *Kernel → Change Kernel… → CRAM*. The default Python kernel does not
> have the semantic digital twin installed.

We will look at worlds through **RViz2** instead of an inline notebook widget — the same tool
you would point at a real robot. Open RViz2 now (`rviz2` in a terminal with the workspace
sourced), and add:

- a **TF** display, with the fixed frame set to `map`,
- a **MarkerArray** display, subscribed to `/semworld/viz_marker`.

Leave both running for the rest of the notebook — every world we build below publishes to the
same topic, so RViz always shows whichever world you last looked at.

Run the cell below. It should print a version and a few `OK` lines.

In [ ]:
import logging
import threading
from collections import Counter
from pathlib import Path
from importlib.resources import files

import numpy as np
import rclpy

import semantic_digital_twin
from semantic_digital_twin.adapters.package_resolver import CompositePathResolver
from semantic_digital_twin.adapters.ros.tf_publisher import TFPublisher
from semantic_digital_twin.adapters.ros.visualization.viz_marker import VizMarkerPublisher
from semantic_digital_twin.adapters.urdf import URDFParser
from semantic_digital_twin.api import BodySpecification, RevoluteConnectionSpecification, RobotSpecification
from semantic_digital_twin.exceptions import ExerciseVerificationFailed
from semantic_digital_twin.reasoning import world_rdr
from semantic_digital_twin.reasoning.world_reasoner import WorldReasoner
from semantic_digital_twin.robots.pr2 import PR2
from semantic_digital_twin.semantic_annotations.semantic_annotations import (
    Door,
    Drawer,
    Dresser,
    Fridge,
    Handle,
    Slider,
    Wardrobe,
)
from semantic_digital_twin.spatial_types.spatial_types import (
    HomogeneousTransformationMatrix,
    Vector3,
)
from semantic_digital_twin.world import World
from semantic_digital_twin.world_description.connections import PrismaticConnection
from semantic_digital_twin.world_description.geometry import Scale

logging.disable(logging.CRITICAL)  # keep the notebook output readable

print("semantic_digital_twin", semantic_digital_twin.__version__)

URDF_DIR = Path(files("semantic_digital_twin")).parent.parent / "resources" / "urdf"
KITCHEN = URDF_DIR / "kitchen.urdf"
print("OK  kitchen URDF:", KITCHEN.name)

# The kitchen references its meshes as package://iai_kitchen/... , which comes from the
# iai_maps ROS package. If this line fails, the ROS workspace was not sourced.
CompositePathResolver().resolve("package://iai_kitchen/meshes/misc/Sink.obj")
print("OK  mesh packages resolve")

# One ROS2 node for the whole notebook. Every world we visualize below publishes onto it.
rclpy.init()
node = rclpy.create_node("semantic_digital_twin_tutorial")
threading.Thread(target=rclpy.spin, args=(node,), daemon=True).start()
print("OK  ROS2 node spinning")


def visualize(world: World, topic_name: str = "/semworld/viz_marker") -> None:
    '''Publish `world` to RViz2. Call once per world; further changes to it (opening a
    door, moving a joint, ...) are pushed automatically, no need to call this again.'''
    TFPublisher(_world=world, node=node)
    VizMarkerPublisher(_world=world, node=node, topic_name=topic_name)


print("OK  visualize() ready — look at RViz2 now")

## 1. What a URDF can and cannot tell a robot

`URDFParser` reads a URDF into a `World`. A `World` is a graph: bodies and regions as
nodes, connections (joints) as edges, with a registry of degrees of freedom.

*(This URDF has a couple of minor XML issues — a `material` tag inside a `collision`
element, an attribute the parser doesn't recognise — so you will see a few `Unknown tag` /
`Unknown attribute` warnings. They are harmless; real-world URDFs are rarely clean.)*

In [ ]:
world = URDFParser.from_file(str(KITCHEN)).parse()

print("bodies     ", len(world.bodies))
print("connections", len(world.connections))
print(Counter(type(c).__name__ for c in world.connections))

Let's look at it.

In [ ]:
visualize(world)

### Now the important part

Ask the world what it *means*:

In [ ]:
print(world.semantic_annotations)

Empty. A world parsed from a file is a purely kinematic model. Nothing in it is a
fridge, a drawer, or a door — those are things *we* read into the link names.

### Exercise 1 — find everything the robot can pull open

You are writing the perception layer for a robot in this kitchen. It needs a list of
drawers. There are two obvious ways to guess, and they are both reasonable.

**Guess A — match the names.** Somebody called them drawers, so search for that:

In [ ]:
by_name = {body.name.name for body in world.bodies if "drawer" in body.name.name.lower()}
print(len(by_name), "bodies matched by name")

**Guess B — match the structure.** A drawer slides, so look for prismatic joints:

In [ ]:
sliding = {c.child.name.name for c in world.connections if isinstance(c, PrismaticConnection)}
print(len(sliding), "bodies sit on a prismatic joint")

Nowhere near the same number. Whatever the truth is, at least one of these guesses is
badly wrong. Look at what Guess A found that Guess B didn't:

In [ ]:
only_named = sorted(by_name - sliding)
handles = [n for n in only_named if n.endswith("_handle")]
rest = [n for n in only_named if n not in handles]

print(len(handles), "of them are handles, e.g.:", handles[:3])
print(len(rest), "of them are something else, e.g.:", rest[:3])

Two different failure modes, both from the same heuristic:

- `sink_area_left_bottom_drawer_handle` and its siblings are **handles**. They matched only
  because the word "drawer" appears in the name of the drawer they belong to — the handle
  itself does not slide, its drawer does.
- The `drawer_oven_right_board*_link` bodies are internal shelves *inside* one particular
  oven drawer, rigidly fixed to its front panel. They are named after the drawer, but they
  are not drawers themselves — they don't have their own joint at all.

And the name-based guess has a deeper problem than being wrong in these particular cases: it
only works because a human happened to name these links in English. Rename them
`link_001 … link_084` — a perfectly legal URDF, and what most CAD exporters give you — and it
returns nothing.

The rest of this tutorial is about getting an answer that does not depend on either luck.

## 2. Saying what things are

A **semantic annotation** attaches meaning to bodies in a world: *this* body is a handle,
*these* bodies together are a drawer.

The library's position is worth stating plainly, because it is a design choice you may want
to argue with. Annotations are inspired by ontologies, but they are **not** an ontology:
there is no OWL, no RDF, no triple store, no separate reasoner. An annotation is an ordinary
Python dataclass, and reasoning is done with Python's type system plus a query language. The
claim is that you get the expressiveness without the impedance mismatch; the cost is that
your knowledge lives in Python rather than in a portable standard.

Here is a real one from the library:

```python
@dataclass(eq=False)
class Drawer(Furniture, HasCaseAsRootBody, HasHandle, HasMechanicalJoint):
    @classproperty
    def hole_direction(self) -> Vector3:
        return Vector3.Z()
```

The mixins *are* the definition: a drawer is furniture, it has a case as its root body, it
has a handle, and it has a mechanical joint. Remember that definition — in §3 it turns out to
be slightly too strict, and that is exactly why the fridge's own case has a drawer the reasoner
cannot see.

Building one of these takes two ingredients you haven't used yet: a way to place bodies in
space, and the `add` method that wires annotations together. Transforms first.

### A quick word on transforms

Every `world_root_T_self=...` you will write below is a `HomogeneousTransformationMatrix` — a
4×4 matrix saying where one frame sits relative to another. The library's convention, used
everywhere: read `A_T_B` as *"the pose of B, expressed in frame A."* (The full story is in
`examples/using_transformations.md`, linked again at the end of this notebook.)

Two things about them are enough for what follows:

- `HomogeneousTransformationMatrix.from_xyz_rpy(x=.., y=.., z=.., roll=.., pitch=.., yaw=..)`
  builds one from a position and Euler angles — everything defaults to `0`.
- Transforms compose with `@`. If you know `world_T_a` and `a_T_b`, then `world_T_a @ a_T_b`
  is `world_T_b` — the pose of `b`, expressed in `world`.

In [ ]:
world_T_a = HomogeneousTransformationMatrix.from_xyz_rpy(x=1.0)
a_T_b = HomogeneousTransformationMatrix.from_xyz_rpy(y=0.5)
world_T_b = world_T_a @ a_T_b

print("world_T_b position:", world_T_b.to_position().to_np().flatten()[:3])

### Exercise 2 — build a transform

Below you will place a handle 0.28 m in front of a drawer's origin (negative `x`). Practice
that transform first.

Build a `HomogeneousTransformationMatrix` with `from_xyz_rpy` that has no rotation and sits
0.28 m in the negative `x` direction. Assign it to `handle_offset`.

In [ ]:
handle_offset: HomogeneousTransformationMatrix = ...

# TODO: build the transform

In [ ]:
# Run this to check your answer.
if handle_offset is ... or not isinstance(handle_offset, HomogeneousTransformationMatrix):
    raise ExerciseVerificationFailed(
        "handle_offset should be a HomogeneousTransformationMatrix."
    )

position = handle_offset.to_position().to_np().flatten()[:3]
expected = np.array([-0.28, 0.0, 0.0])
if not np.allclose(position, expected, atol=1e-6):
    raise ExerciseVerificationFailed(f"Expected position {expected}, got {position}.")

print("Correct.")

### Building a drawer

`create_with_new_body_in_world` spawns the annotation, a body, and its geometry in one call.
Then `add` wires the parts together — one method, routed by type.

In [ ]:
drawer_world = World.create_with_root_body()

with drawer_world.modify_world():
    drawer = Drawer.create_with_new_body_in_world(
        name="drawer",
        scale=Scale(0.5, 0.5, 0.4),
        world=drawer_world,
        world_root_T_self=HomogeneousTransformationMatrix(),
    )
    handle = Handle.create_with_new_body_in_world(
        name="drawer_handle",
        scale=Scale(0.05, 0.25, 0.1),
        world_root_T_self=HomogeneousTransformationMatrix.from_xyz_rpy(x=-0.25),
        world=drawer_world,
    )
    slider = Slider.create_with_new_body_in_world(
        name="drawer_slider",
        world_root_T_self=HomogeneousTransformationMatrix(),
        world=drawer_world,
        parent_connection_specification=Slider.parent_connection_specification(
            axis=Vector3.NEGATIVE_X()
        ),
    )

    # One method. It matches each part against the typed part-whole fields of the whole.
    drawer.add(handle)   # -> drawer.handle           (single-valued field)
    drawer.add(slider)   # -> drawer.mechanical_joint (single-valued field)

print("drawer.handle           is handle:", drawer.handle is handle)
print("drawer.mechanical_joint is slider:", drawer.mechanical_joint is slider)

Note what `add` did *not* need: no `parent=`, no `child=`, no joint declaration. The part's
type was enough to decide both which field it belongs in and where it mounts in the
kinematic tree. You can see the tree it built:

In [ ]:
drawer_world.visualize_world_structure()

`visualize_world_structure` is the most useful debugging tool in the library — it draws the
kinematic tree as the robot actually sees it. (Try it on `world`, the kitchen, if you like: 84
bodies make a very wide image.)

### The payoff

The annotation is not a label sitting beside the geometry — it is wired into it. `Drawer`
has a `mechanical_joint`, so a drawer can be opened. Here is what you built:

In [ ]:
visualize(drawer_world, topic_name="/semworld/viz_marker")

`add` made the handle a kinematic child of the drawer, so it will always move with the
drawer's front. There is nothing else in this scene for the drawer to slide out of yet, so
you cannot see that for real — but you are about to build a case for it to open out of, which
is exactly what "open the fridge door" needs one section from now: one line of code,
`mechanical_joint.position = ...` (or, as you'll see in §4, the equivalent
`root.parent_connection.position = ...`), and the geometry follows.

### Exercise 3 — build a dresser

Build a `Dresser` containing a drawer like the one you just built by hand. You need a
`Dresser`, a `Drawer`, a `Handle` and a `Slider`, wired together with `add` exactly as above.

One thing is new: a `Dresser` is a case, and a case only opens on one side —
`Dresser.hole_direction` says which. Get the slider's axis to match, or the drawer will slide
into the back of the case instead of out through the front.

Then open the drawer and visualize.

In [ ]:
dresser_world: World = ...
dresser: Dresser = ...
dresser_drawer: Drawer = ...
dresser_handle: Handle = ...
dresser_slider: Slider = ...

# TODO: create a World, then inside world.modify_world():
#         - a Dresser
#         - a Drawer
#         - a Handle, offset from the drawer's origin (see Exercise 2)
#         - a Slider, with the axis that matches Dresser.hole_direction
#       wire them together with add(), exactly as in the drawer you built above.

# TODO: open the drawer, then visualize it

In [ ]:
# Run this to check your answer.
if dresser is ... or not isinstance(dresser, Dresser):
    raise ExerciseVerificationFailed("dresser should be a Dresser.")
if dresser_drawer is ... or not isinstance(dresser_drawer, Drawer):
    raise ExerciseVerificationFailed("dresser_drawer should be a Drawer.")
if dresser_drawer.handle is not dresser_handle:
    raise ExerciseVerificationFailed(
        "Use dresser_drawer.add(dresser_handle) to attach the handle."
    )
if dresser_drawer.mechanical_joint is not dresser_slider:
    raise ExerciseVerificationFailed(
        "Use dresser_drawer.add(dresser_slider) to attach the slider."
    )
if dresser_drawer not in dresser.drawers:
    raise ExerciseVerificationFailed(
        "Use dresser.add(dresser_drawer) so the dresser owns the drawer."
    )
if dresser_drawer.mechanical_joint.position == 0:
    raise ExerciseVerificationFailed(
        "Open the drawer — set dresser_drawer.mechanical_joint.position to something nonzero."
    )

slider_axis = dresser_drawer.mechanical_joint.root.parent_connection.axis.to_np().flatten()[:3]
hole_direction = Dresser.hole_direction.to_np().flatten()[:3]
if slider_axis @ hole_direction <= 0:
    raise ExerciseVerificationFailed(
        "The slider's axis should point the same way as Dresser.hole_direction, or opening "
        "the drawer pushes it into the back of the case instead of out the front."
    )

print("Correct.")
visualize(dresser_world)

### One detail that matters later

Annotations are declared `@dataclass(eq=False)`. That looks like boilerplate, but it is
load-bearing: the base class defines equality and hashing **structurally**, over the type and
the bodies referenced. Two separately constructed `Handle` objects on the same body are the
same handle:

In [ ]:
first = Handle(root=handle.root)
second = Handle(root=handle.root)

print("different objects:", id(first) != id(second))
print("but equal        :", first == second)
print("and same hash    :", hash(first) == hash(second))

Without this, the reasoner in §3 could not tell a newly inferred annotation from one it had
already found, and every run would pile up duplicates.

## 3. Not annotating by hand

Hand-annotating a dresser took twenty lines. The kitchen has 84 bodies, and a real kitchen
dataset has thousands. This does not scale, and it is not supposed to.

`WorldReasoner` applies a body of rules to a raw world and infers the annotations itself. It
runs on the kitchen we loaded in §1 — the plain URDF, with nothing added.

In [ ]:
if "world" not in globals():
    world = URDFParser.from_file(str(KITCHEN)).parse()

reasoner = WorldReasoner(world)
inferred = reasoner.reason()["semantic_annotations"]

print(f"{len(inferred)} annotations inferred\n")
print(Counter(type(a).__name__ for a in inferred))

From a file that contained none of those words as *concepts*, the reasoner produced handles,
drawers, doors — and a `Fridge`. So what does it say about Exercise 1?

In [ ]:
drawers = world.get_semantic_annotations_by_type(Drawer)

print(f"{len(drawers)} drawers\n")
for d in drawers:
    print("   ", d.root.name.name)

**Fourteen** — out of 15 bodies that actually slide. The reasoner is being conservative:
it only claims what its rules actually support.

### Why does it think that is a drawer?

The reasoner is not a network. It can show its work.

In [ ]:
from krrood.entity_query_language.explanation.explanation import explain_inference
from krrood.entity_query_language.verbalization.pipeline import verbalize_expression

explanation = explain_inference(drawers[0])

print(explanation.get_satisfied_conditions_as_string())

In [ ]:
print(verbalize_expression(explanation.query_root))

Read that rule closely, because it is doing something neither heuristic in §1 could:

> If there's a FixedConnection whose parent is the child of a PrismaticConnection, there's a
> Handle whose root is the child of the FixedConnection, then there's a Drawer whose root is
> the parent of the FixedConnection, and whose handle is the Handle.

It is **structural**: "a body that slides, with a handle rigidly attached to it." Rename every
link to `link_042` and this rule still fires.

The rules are not a black box and not a trained artifact. They are generated Python, checked
into the repository next to the code, so they are reviewed, versioned and migrated like
everything else:

In [ ]:
print(Path(world_rdr.__file__).parent)
for f in sorted(Path(world_rdr.__file__).parent.glob("*.py")):
    print("   ", f.name)

### The one it missed

Fifteen bodies slide. Fourteen became drawers. Which one didn't, and why?

In [ ]:
sliding_but_not_drawer = sliding - {d.root.name.name for d in drawers}
print(sliding_but_not_drawer)

In [ ]:
def children_of(body_name):
    body = world.get_body_by_name(body_name)
    children = body.child_kinematic_structure_entities
    if not children:
        return "   (nothing attached)"
    return "\n".join(
        f"   {c.name.name}  via {type(c.parent_connection).__name__}" for c in children
    )

for name in list(sliding_but_not_drawer) + ["sink_area_left_bottom_drawer_main"]:
    print(f"{name}:")
    print(children_of(name))
    print()

There it is. `oven_area_area_right_drawer_main` slides, like every other drawer — but
compare its children to `sink_area_left_bottom_drawer_main`'s, a drawer the reasoner *did*
find. The good drawer has a `Handle` fixed to its front. This one has four internal shelves
and their guard rails, fixed to its front — but no handle anywhere.

Recall the definition from §2:

```python
class Drawer(Furniture, HasCaseAsRootBody, HasHandle, HasMechanicalJoint):
```

The rule implements exactly that: a drawer *has a handle*. This one slides, it sits inside a
cabinet, a robot could open it by pulling on its front panel — but no handle was modelled, so
the rule cannot see it. This is not a naming problem, and no amount of better string matching
would help. The definition is a little too strict for this data.

Keep that in mind for §4: it is exactly the kind of gap you want to know about *before* you
trust a reasoner to tell a robot what it can and cannot open.

## 4. The fridge, and the door in the way of the milk

Here is the whole point of the exercise. "Take the milk out of the fridge" needs the robot to
know, at minimum: which body is the fridge, that it has a door, which way the door opens, how
far, and where its handle is. All of that is now sitting in the world as typed annotations —
found automatically, not written by hand.

In [ ]:
fridge = world.get_semantic_annotations_by_type(Fridge)[0]

print("fridge body :", fridge.root.name.name)
print("fridge type :", type(fridge).__name__, "is a", type(fridge).__mro__[1].__name__)
print("doors       :", [d.root.name.name for d in fridge.doors])

door = fridge.doors[0]
print("door handle :", door.handle.root.name.name)

hinge = door.root.parent_connection
print("hinge joint :", type(hinge).__name__)
print("hinge range :", hinge.dof.limits.lower.position, "->", hinge.dof.limits.upper.position)

`Fridge` is a `Cabinet` — the same family as `Dresser`, which you built by hand in §2 — so
everything you learned there applies here without change. `fridge.doors[0].root.parent_connection`
is the same kind of object as the drawer's slider connection: a real joint, with real limits,
that you move by writing to `.position`.

*(The furniture unit standing next to the appliance, `fridge_area`, has its own drawer —
that's a separate piece of furniture the reasoner also found, and not part of the `Fridge`
annotation itself. The door above belongs to `iai_fridge_main`, the appliance.)*

Nothing is reachable, grasped, or moved by a gripper in this notebook — that is a manipulation
problem for later. But the door between the robot and the milk is not a manipulation problem
yet; it's a *world* problem, and it is one you can already solve:

### Exercise 4 — open the fridge

1. Get `fridge.doors[0]`'s driving connection the way `hinge` was computed above, and assign
   it to `fridge_hinge`.
2. Set `fridge_hinge.position` to something inside its limits, closer to the upper limit than
   to `0`, so the door is clearly open.
3. Visualize `world` and check RViz — the fridge door should have swung open.

In [ ]:
fridge_hinge = ...

# TODO: get the fridge door's connection (see how `hinge` was computed above)
# TODO: open it, then visualize `world`

In [ ]:
# Run this to check your answer.
expected_connection = fridge.doors[0].root.parent_connection
if fridge_hinge is not expected_connection:
    raise ExerciseVerificationFailed(
        "fridge_hinge should be fridge.doors[0].root.parent_connection."
    )

lower = fridge_hinge.dof.limits.lower.position
upper = fridge_hinge.dof.limits.upper.position
if not (lower <= fridge_hinge.position <= upper):
    raise ExerciseVerificationFailed(f"fridge_hinge.position must be within {lower} to {upper}.")
if fridge_hinge.position < 0.6 * upper:
    raise ExerciseVerificationFailed(
        "Open the door further — set fridge_hinge.position closer to the upper limit."
    )

print("Correct. Check RViz: the fridge door should be open.")
visualize(world)

Notice you did not need to call `visualize` again for the door to move in RViz — opening
the door is a *state* change, not a *model* change, and the `TFPublisher` you registered back
in §1 pushes state changes automatically. `visualize` only needs to be called once per world,
the first time you build or load it.

That single line, `fridge_hinge.position = ...`, is also everything "open the fridge" needs
from the world model. What comes after — planning a collision-free reach to the handle,
grasping it, pulling it through the hinge's range — builds on exactly this representation,
one layer up, and is where the next notebook in this series picks up.

## 5. Robots

Everything so far has lived in a static kitchen. A robot is just another URDF — parsed the
same way, and merged into the *same* world the kitchen already lives in, not a separate one.
`spawn` parses the robot's URDF, merges it into `world`, and annotates it while doing so —
`pr2` below is itself a semantic annotation, decomposing into typed parts (arms, torso,
grippers, ...) the same way `fridge` decomposes into doors and handles.

This notebook stops at *bringing the robot into the world*. Moving its joints, planning a
reach to the now-open fridge, and grasping the milk are exactly the "manipulation" layer this
tutorial has been deferring since the introduction — that's the next notebook.

In [ ]:
if "world" not in globals():
    world = URDFParser.from_file(str(KITCHEN)).parse()

pr2 = RobotSpecification(
    semantic_annotation_type=PR2,
    world_T_odom=HomogeneousTransformationMatrix.from_xyz_rpy(x=0.3, y=-2.3),
).spawn(world)

print("pr2 is a semantic annotation of type:", type(pr2).__name__)
print("bodies in world now:", len(world.bodies))

visualize(world)

*(The PR2's own URDF trips a couple of harmless parser warnings, same as the kitchen's
`material`-in-`collision` tag back in §1.)*

Check RViz: the PR2 should now be standing in the kitchen, in front of the still-open fridge
door from §4 — the same world, carried forward.

## 6. On your own

Put the whole loop together, unassisted, on a second, smaller kitchen: parse a URDF, run the
reasoner, find the fridge, open its door.

### Exercise 5 — the other kitchen

`kitchen-small.urdf` sits next to `kitchen.urdf` in the same directory. It is a different
model of a similar kitchen — its dishwasher and oven doors happen to have their joint limits
set to `0` in this particular file (a modelling gap, same spirit as the handle-less drawer in
§3), so the fridge door is the one reliably worth opening.

1. Parse `URDF_DIR / "kitchen-small.urdf"` into a new `World`, assign it to `small_world`.
2. Run a `WorldReasoner` over it.
3. Get its `Fridge` annotation, assign it to `small_fridge`.
4. Open `small_fridge.doors[0]`'s connection to something past the midpoint of its range —
   the same pattern as Exercise 4 — and assign the connection to `small_hinge`.
5. Visualize `small_world` and check RViz.

In [ ]:
small_world: World = ...
small_fridge: Fridge = ...
small_hinge = ...

# TODO: 1. parse kitchen-small.urdf into small_world
# TODO: 2. run WorldReasoner(small_world).reason()
# TODO: 3. small_fridge = small_world.get_semantic_annotations_by_type(Fridge)[0]
# TODO: 4. open small_fridge.doors[0]'s connection, assign it to small_hinge
# TODO: 5. visualize(small_world)

In [ ]:
# Run this to check your answer.
if small_world is ... or not isinstance(small_world, World):
    raise ExerciseVerificationFailed("small_world should be a World.")
if small_fridge is ... or not isinstance(small_fridge, Fridge):
    raise ExerciseVerificationFailed(
        "small_fridge should be the Fridge annotation found by the reasoner."
    )

expected_connection = small_fridge.doors[0].root.parent_connection
if small_hinge is not expected_connection:
    raise ExerciseVerificationFailed(
        "small_hinge should be small_fridge.doors[0].root.parent_connection."
    )

lower = small_hinge.dof.limits.lower.position
upper = small_hinge.dof.limits.upper.position
if small_hinge.position < lower + 0.6 * (upper - lower):
    raise ExerciseVerificationFailed(
        "Open the door further — set small_hinge.position closer to the upper limit."
    )

print("Correct. Check RViz: the second kitchen's fridge door should be open.")
visualize(small_world, topic_name="/semworld/viz_marker")

If you have time left, try the same pattern — `get_semantic_annotations_by_type`, then
`.root.parent_connection.position = ...` — on `small_fridge.drawers`, or on the `Door` and
`Drawer` annotations the reasoner found elsewhere in `small_world`. It is the same three
lines every time, on every piece of furniture, because the annotation is what made "open it"
a well-defined operation in the first place.

### Cleanup

When you are done, shut down the ROS2 node cleanly.

In [ ]:
node.destroy_node()
rclpy.shutdown()

## 7. Where to go next

What we did: took a URDF that knew only geometry, gave it a vocabulary of typed concepts, had
a rule base infer those concepts automatically, found a case the rules got wrong, and used the
result to open the one thing standing between a robot and a carton of milk.

The thing worth taking away is the last part of §3. The heuristics in §1 were wrong and gave
you nothing to work with. The reasoner was also incomplete — but it could tell you exactly
what rule produced each answer, which is what made the gap findable in a few lines instead of
a mystery.

What we deliberately left out — reaching for the handle, planning a collision-free path,
grasping the milk, querying the world with the Entity Query Language — is exactly where the
next notebook in this series picks up, on top of the same world model you just built.

The library ships a full Jupyter Book under
`cognitive_robot_abstract_machine/semantic_digital_twin/doc/` (17 worked examples, concept
chapters, and self-assessment quizzes):

| Topic | Guide |
|---|---|
| Transforms and the `A_T_B` convention | `examples/using_transformations.md` |
| Declarative world building | `examples/building_worlds_with_specifications.md` |
| Visualizing worlds (RViz2 and simulation) | `examples/visualizing_worlds.md` |
| The Entity Query Language | `examples/entity_query_language.md` |
| Regions and supporting surfaces | `examples/regions.md` |
| Saving annotated worlds to SQL | `examples/persistence_of_annotated_worlds.md` |
| Physics simulation (MuJoCo) | `examples/physics_simulators.md` |
| Adding a new robot | `examples/adding_robots.md` |
| Free-space decomposition and path planning | `examples/graph_of_convex_sets.md` |
| Loading RoboCasa / ProcTHOR / PartNet scenes | `doc/datasets.md` |

To convert any of them into a runnable notebook:

```bash
jupytext --to notebook cognitive_robot_abstract_machine/semantic_digital_twin/examples/regions.md
```

The predecessor to this tutorial, on writing the URDF itself, is
[EASE Fall School 2024 — Creating an Environment URDF](https://github.com/IntEL4CoRo/ease_fall_school_2024).